> **公开学习副本**：保留教学代码与解释，已清除运行输出、执行计数、嵌入附件及个人元数据。完整执行原件和H02交接包保留在本地/组内私有空间。不要把本副本误认为团队已验收的新数据版本。
>
> 数据来自Olist；基础教学材料参考[CercaTrovato/olist-repurchase-analysis](https://github.com/CercaTrovato/olist-repurchase-analysis)，保留上游代码与数据许可。仅在本地有原始数据时运行；输出可能包含逐行客户数据，保持在被Git忽略的outputs目录。源码中的定义及限制仍应阅读。
> 岗位学习课：输出隔离到outputs/public_learning/，版本带public-learning，动态选取示例客户。不能替换已发给团队的H02_candidate_v2。按02→03→04运行。

# 1 号第一步 从原始记录到订单事实表与历史客户快照

这份 Notebook 对应岗位操作第 1 项。目标是让你能追踪“某条商品明细怎样形成订单金额，订单怎样归到买家，再怎样进入某个快照的历史统计”。所有代码在本文件中逐格展开，运行输出保存在文件里。

本次采用团队现行工作口径：批准时间存在，商品金额大于零；历史使用快照前六个自然月。用 **2018-01-01** 这个已探索过的日期演示客户聚合。这个日期由助手为本次教学选择，正式训练、验证、最终评价方案仍待团队核定。

本轮完成订单和客户键检查、明细粒度检查、金额汇总、两次连接、资格筛选、一个快照的历史客户聚合，以及输出追溯。未来 90 天标签、完整每日／每月覆盖检查、最终评价标签保管和正式 H02 封版在后续步骤完成。本次历史客户文件没有 `y`。

## H02 的五类文件从哪里产生

| 最终交付 | 依赖的操作 | 本次进度 |
|---|---|---|
| snapshot_features.csv | 第 1 项确定粒度、资格和客户聚合；第 2 项完善历史特征；第 3 项核时点 | 生成学习用历史聚合基础，正式字段与时间分组待核 |
| snapshot_labels.csv | 第 2 项生成未来目标；第 3 项核边界并按开放范围分享 | 后续步骤 |
| data-manifest.json | 三项操作共同提供来源、版本、文件哈希与规则 | 生成 step1-manifest.json 记录本次范围 |
| data-dictionary.md | 原始字段、聚合公式、特征和标签定义 | 生成本步字段说明 |
| coverage-and-cohorts.csv | 第 1 项筛选人数；第 2 项覆盖和正例；第 3 项成熟性检查 | 本次只生成筛选及历史人数，不计算正例 |

操作时从上至下运行。若内核重启，应重新从首格开始。你可以先读每格上方的问题，再运行，再用下方说明解释输出。

## 1 定位项目和本次输出

先确认程序在使用哪套 Python、哪个原始目录，以及输出写到哪里。输出使用独立目录 `outputs/public_learning/role01_step1_v1`。重复执行会更新本步生成文件。

`SNAPSHOT_DATE` 定义演示快照，`HISTORY_MONTHS` 定义自然月数。六个月和 180 天在日历上可能不同。

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

search_locations = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in search_locations if (path / 'data/source-manifest.json').is_file()),
    None
)
assert PROJECT_ROOT is not None, '请从本仓库根目录或其中的 notebooks 目录启动。'
DATA_DIR = PROJECT_ROOT / 'data/raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/public_learning/role01_step1_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_VERSION = 'role01-step1-public-learning'
SNAPSHOT_DATE = pd.Timestamp('2018-01-01')
HISTORY_MONTHS = 6
HISTORY_START = SNAPSHOT_DATE - pd.DateOffset(months=HISTORY_MONTHS)
NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks/02-role01-order-fact-and-history.ipynb'

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)
print('Python', sys.version.split()[0], '| pandas', pd.__version__)
print('原始数据', DATA_DIR)
print('本次输出', OUTPUT_DIR)
print('历史区间', HISTORY_START, '≤ 批准时间 <', SNAPSHOT_DATE)
print('用途：已探索日期的历史聚合教学；正式时间方案待核定。')

## 2 校验原件并只读取本步需要的三张表

本步需要订单、客户、商品明细。支付、评价和坐标对基础 RFM 不是必需输入。这里核验 ZIP 和本次实际读取的三个 CSV 指纹，并记录本地协议和参考脚本版本。

哈希相同证明输入文件与来源清单一致。接下来还要检查键和业务数据本身。ID 用字符串读取，邮编前缀也保留字符串形式。

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

source_manifest = json.loads((PROJECT_ROOT / 'data/source-manifest.json').read_text(encoding='utf-8'))
archive_sha256 = file_sha256(PROJECT_ROOT / 'data/olist-v2.zip')
assert archive_sha256 == source_manifest['zip_sha256'], 'ZIP 指纹与来源清单不符'
file_names = {
    'orders': 'olist_orders_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
}
expected_hashes = {item['name']: item['sha256'] for item in source_manifest['files']}
source_checks = []
for table_name, file_name in file_names.items():
    actual_hash = file_sha256(DATA_DIR / file_name)
    source_checks.append({'table': table_name, 'file': file_name,
                          'sha256': actual_hash, 'matches_manifest': actual_hash == expected_hashes[file_name]})
source_checks = pd.DataFrame(source_checks)
display(source_checks)
assert source_checks['matches_manifest'].all(), 'CSV 指纹不一致，先核查输入'

orders_raw = pd.read_csv(DATA_DIR / file_names['orders'], dtype={'order_id': 'string', 'customer_id': 'string'})
customers = pd.read_csv(DATA_DIR / file_names['customers'], dtype={
    'customer_id': 'string', 'customer_unique_id': 'string', 'customer_zip_code_prefix': 'string'})
items = pd.read_csv(DATA_DIR / file_names['items'], dtype={
    'order_id': 'string', 'product_id': 'string', 'seller_id': 'string'})

reference_paths = ['docs/role01/protocol-reference.md', 'docs/role01/role-01-reference.md', 'scripts/experiment_common.py']
reference_hashes = {name: file_sha256(PROJECT_ROOT / name) for name in reference_paths}

## 3 键检查和处理台账

orders 应该一行一订单，customers 一行一个 customer_id，items 一行一个 order_id 与 order_item_id 的组合。`customer_unique_id` 可以重复，说明一个买家有多个订单侧客户编号。

下面的小函数只负责记录检查结果。台账中的 `excluded_rows` 表示主动排除的行数；`collapsed_rows` 表示聚合把多行折叠后的行数差；`unmatched_rows` 表示连接未匹配行数。三者不能混用。没有执行连接的步骤用空值表示未匹配检查不适用。

In [ ]:
stage_log = []

def record_stage(name, frame, key, grain, *, unmatched_rows=None,
                 excluded_rows=0, collapsed_rows=0, note=''):
    row = {
        'stage': name, 'grain': grain, 'rows': len(frame), 'key': '+'.join(key),
        'distinct_keys': len(frame[key].drop_duplicates()),
        'null_key_rows': int(frame[key].isna().any(axis=1).sum()),
        'duplicate_extra_rows': int(frame.duplicated(key).sum()),
        'key_unique': not frame.duplicated(key).any(),
        'buyers': int(frame['customer_unique_id'].nunique()) if 'customer_unique_id' in frame else None,
        'unmatched_rows': unmatched_rows, 'excluded_rows': excluded_rows,
        'collapsed_rows': collapsed_rows, 'note': note,
    }
    stage_log.append(row)
    display(pd.DataFrame([row]))

record_stage('01_read_orders', orders_raw, ['order_id'], '订单')
record_stage('02_read_customers', customers, ['customer_id'], '订单侧客户编号')
record_stage('03_read_items', items, ['order_id', 'order_item_id'], '商品明细')

assert orders_raw['order_id'].notna().all() and orders_raw['order_id'].is_unique
assert customers['customer_id'].notna().all() and customers['customer_id'].is_unique
assert orders_raw['customer_id'].notna().all() and orders_raw['customer_id'].is_unique
assert customers['customer_unique_id'].notna().all()
assert items[['order_id', 'order_item_id']].notna().all().all()
assert not items.duplicated(['order_id', 'order_item_id']).any()
print('订单侧客户编号数', customers['customer_id'].nunique())
print('跨订单匿名买家数', customers['customer_unique_id'].nunique())

你应看到 99,441 笔订单、99,441 个订单侧客户记录、112,650 条商品明细。匿名买家有 96,096 个。检查每一行台账的粒度，先确认唯一键与它相符，再看数量。

## 4 在连接前检查左右两侧的缺口

用 `isin` 检查编号是否存在于另一张表。`~` 反转真假值，所以 `~orders_raw.order_id.isin(items.order_id)` 找出没有明细的订单。

既要看“订单没有明细”，也要看“明细找不到订单”。本次要求主干键无空值、无孤立客户或孤立明细；如失败，程序先停止，让问题暴露出来。

In [ ]:
relation_checks = pd.DataFrame([
    {'check': '订单 customer_id 找不到客户', 'unit': '订单行',
     'count': int((~orders_raw['customer_id'].isin(customers['customer_id'])).sum())},
    {'check': '客户 customer_id 没有对应订单', 'unit': '客户记录行',
     'count': int((~customers['customer_id'].isin(orders_raw['customer_id'])).sum())},
    {'check': '订单 order_id 没有商品明细', 'unit': '订单行',
     'count': int((~orders_raw['order_id'].isin(items['order_id'])).sum())},
    {'check': '商品明细 order_id 找不到订单', 'unit': '明细行',
     'count': int((~items['order_id'].isin(orders_raw['order_id'])).sum())},
])
display(relation_checks)
assert orders_raw['customer_id'].isin(customers['customer_id']).all()
assert items['order_id'].isin(orders_raw['order_id']).all()

amount_input_check = pd.DataFrame({
    'field': ['price', 'freight_value'],
    'missing': [int(items['price'].isna().sum()), int(items['freight_value'].isna().sum())],
    'negative': [int(items['price'].lt(0).sum()), int(items['freight_value'].lt(0).sum())],
})
display(amount_input_check)
assert items[['price', 'freight_value']].notna().all().all(), '未知明细金额需要先核查，不能静默求和'
assert items[['price', 'freight_value']].ge(0).all().all(), '出现负金额，先明确事件与金额规则'

本数据有 775 笔订单没有商品明细。稍后左连接会保留它们，商品金额显示为缺失。此处没有删除 775 笔订单。

## 5 商品明细按订单聚合

目标是一行一订单。`amount` 把同一订单的 price 相加，`freight` 把同一订单的运费相加，`item_rows` 数明细条数。使用与仓库参考脚本相同的三个聚合字段。

聚合后的行数会减少，因为多条明细折叠为一个订单。只要明细数与金额核对一致，这种减少属于粒度转换。

In [ ]:
item_order = items.groupby('order_id', as_index=False).agg(
    amount=('price', 'sum'),
    freight=('freight_value', 'sum'),
    item_rows=('order_item_id', 'size'),
)
record_stage('04_aggregate_items', item_order, ['order_id'], '订单商品汇总',
             collapsed_rows=len(items)-len(item_order),
             note='按订单折叠明细；排除量为 0')

amount_reconciliation = []
for source_column, target_column in [('price', 'amount'), ('freight_value', 'freight')]:
    before = float(items[source_column].sum())
    after = float(item_order[target_column].sum())
    assert np.isclose(before, after, rtol=1e-12, atol=1e-7)
    amount_reconciliation.append({'stage': 'items_to_item_order', 'measure': target_column,
                                  'before': before, 'after': after, 'difference': after-before})
assert int(item_order['item_rows'].sum()) == len(items)
display(item_order.head())
display(pd.DataFrame(amount_reconciliation).round(4))

本数据会得到 98,666 行订单商品汇总，商品总额 13,591,643.70 BRL、运费总额 2,251,909.54 BRL。订单表仍有 99,441 笔，因为另外 775 笔缺少明细。

## 6 订单左连接商品汇总

两边用 `order_id` 对齐，`validate='one_to_one'` 检查两边订单号唯一。`indicator='item_match'` 生成匹配标志，`left_only` 是只在订单侧出现的记录。左连接不会产生 right_only，因此右侧孤立键已经在第 4 节另查。

In [ ]:
order_with_items = orders_raw.merge(
    item_order, on='order_id', how='left',
    validate='one_to_one', indicator='item_match'
)
item_unmatched = int(order_with_items['item_match'].eq('left_only').sum())
record_stage('05_join_order_items', order_with_items, ['order_id'], '订单',
             unmatched_rows=item_unmatched, note='左连接保留无商品明细订单，金额保持 NA')
display(order_with_items['item_match'].value_counts().to_frame('orders'))
assert len(order_with_items) == len(orders_raw)
assert set(order_with_items['order_id']) == set(orders_raw['order_id'])
assert order_with_items.loc[order_with_items['item_match'].eq('left_only'), 'amount'].isna().all()
assert np.isclose(order_with_items['amount'].sum(), items['price'].sum(), rtol=1e-12, atol=1e-7)
assert np.isclose(order_with_items['freight'].sum(), items['freight_value'].sum(), rtol=1e-12, atol=1e-7)

## 7 按 customer_id 接上匿名买家编号

订单当前已经有 customer_id。接客户表，是为了取得 customer_unique_id 和客户州。本数据两侧 customer_id 均唯一，因此使用 one_to_one。如果换成其他数据集，需要重新验证这个关系。

此时仍是一行一订单。customer_unique_id 在订单表重复是正常现象，表示同一匿名买家有多笔订单。

In [ ]:
order_fact = order_with_items.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id', how='left', validate='one_to_one', indicator='customer_match'
)
customer_unmatched = int(order_fact['customer_match'].eq('left_only').sum())
record_stage('06_join_customers', order_fact, ['order_id'], '订单',
             unmatched_rows=customer_unmatched, note='增加买家编号和州，不改变订单粒度')
display(order_fact['customer_match'].value_counts().to_frame('orders'))
assert len(order_fact) == len(orders_raw)
assert customer_unmatched == 0
assert order_fact['customer_unique_id'].notna().all()
for measure in ['amount', 'freight']:
    before = float(item_order[measure].sum())
    after = float(order_fact[measure].sum())
    assert np.isclose(before, after, rtol=1e-12, atol=1e-7)
    amount_reconciliation.append({'stage': 'after_two_joins', 'measure': measure,
                                  'before': before, 'after': after, 'difference': after-before})
display(order_fact[['order_id','customer_id','customer_unique_id','amount','freight','item_rows']].head())

## 8 用一笔多明细订单和固定随机样本复核

先选一笔多明细订单，将它的原始明细与订单汇总放在一起。你应能把 price、freight_value 手动相加，与 amount、freight 对上。

再固定随机种子抽五笔有明细的订单。每笔从原始 items 重算金额与条数。这是本步连接核查，完整的未来标签边界核查将在后续任务执行。

In [ ]:
example_order_id = item_order.loc[item_order['item_rows'].gt(1)].sort_values('order_id')['order_id'].iloc[0]
example_items = items.loc[items['order_id'].eq(example_order_id)].copy()
example_fact = order_fact.loc[order_fact['order_id'].eq(example_order_id)].copy()
display(example_items[['order_id','order_item_id','price','freight_value']])
display(example_fact[['order_id','amount','freight','item_rows','customer_unique_id']])

random_order_ids = item_order.sample(n=5, random_state=6400)['order_id'].tolist()
order_trace_checks = []
for order_id in [example_order_id, *random_order_ids]:
    detail = items.loc[items['order_id'].eq(order_id)]
    fact = order_fact.loc[order_fact['order_id'].eq(order_id)].iloc[0]
    amount_ok = bool(np.isclose(detail['price'].sum(), fact['amount'], rtol=1e-12, atol=1e-7))
    freight_ok = bool(np.isclose(detail['freight_value'].sum(), fact['freight'], rtol=1e-12, atol=1e-7))
    rows_ok = len(detail) == fact['item_rows']
    assert amount_ok and freight_ok and rows_ok
    order_trace_checks.append({'order_id': order_id, 'amount_matches': amount_ok,
                               'freight_matches': freight_ok, 'item_rows_match': rows_ok})
display(pd.DataFrame(order_trace_checks))

## 9 把批准时间转为日期并明确合格事件

现在才应用任务资格。批准时间存在且商品金额大于零，就构成当前工作规则下的合格事件。最终 order_status 保留作诊断，未来交付与评分不进入本次历史客户特征。

先检查日期解析。`errors='coerce'` 会把无法解析的文字变成 NaT，因此必须核对非空原值是否解析失败。

每笔不合格订单只记录一个按优先顺序判定的主排除原因，避免同一订单重复计入多个排除阶段。原始订单事实表保留全部订单，另建 eligible_orders 供历史聚合。

In [ ]:
raw_approval = order_fact['order_approved_at'].copy()
order_fact['order_approved_at'] = pd.to_datetime(raw_approval, errors='coerce')
parse_failures = int((raw_approval.notna() & order_fact['order_approved_at'].isna()).sum())
assert parse_failures == 0, f'批准时间存在 {parse_failures} 条解析失败'
order_fact['missing_item_amount'] = order_fact['amount'].isna()
order_fact['eligible_order'] = order_fact['order_approved_at'].notna() & order_fact['amount'].gt(0)
order_fact['exclusion_reason'] = 'included'
order_fact.loc[order_fact['amount'].le(0), 'exclusion_reason'] = 'nonpositive_item_amount'
order_fact.loc[order_fact['amount'].isna(), 'exclusion_reason'] = 'missing_item_amount'
order_fact.loc[order_fact['order_approved_at'].isna(), 'exclusion_reason'] = 'missing_approval'

approved_orders = order_fact.loc[order_fact['order_approved_at'].notna()].copy()
record_stage('07_require_approval', approved_orders, ['order_id'], '订单',
             excluded_rows=len(order_fact)-len(approved_orders), note='排除量单位为订单')
eligible_orders = approved_orders.loc[approved_orders['amount'].gt(0)].copy()
record_stage('08_require_positive_amount', eligible_orders, ['order_id'], '订单',
             excluded_rows=len(approved_orders)-len(eligible_orders), note='在已批准订单中进一步筛金额')

excluded_orders = order_fact.loc[~order_fact['eligible_order'],
    ['order_id','customer_unique_id','order_approved_at','amount','exclusion_reason']].copy()
display(excluded_orders.groupby('exclusion_reason').size().rename('excluded_orders').to_frame())
display(order_fact.groupby(['eligible_order','order_status']).size().rename('orders').reset_index())
assert len(eligible_orders) + len(excluded_orders) == len(order_fact)

本版本预期过程为 99,441 → 99,281 → 98,652。先排除 160 笔无批准时间订单，再排除其中 629 笔无商品金额订单。

为什么不是再减 775？因为 775 笔缺明细订单中，有 146 笔已经在批准时间步骤排除。这个顺序记录可以避免重复计算排除量。

合格事件定义不等于最后成功交付。若团队改成“最终成功交付复购”，需要同时重审事件日期、标签成熟和特征可用性。

## 10 取演示快照的六个月历史

演示区间为 **2017-07-01 ≤ order_approved_at < 2018-01-01**。这里只使用历史订单构造客户名单。

区间外订单保留在事实表中。它们只是在当前快照的历史聚合中不使用，不能称为全局坏数据。下面分别列出区间之前和快照及之后的数量。

这一格没有证明源系统连续完整采集。正式可用性还要在第 2 项执行每日／每月覆盖检查。

In [ ]:
history_mask = eligible_orders['order_approved_at'].ge(HISTORY_START) & eligible_orders['order_approved_at'].lt(SNAPSHOT_DATE)
history_orders = eligible_orders.loc[history_mask].sort_values(['order_approved_at','order_id']).copy()
history_range_counts = pd.DataFrame([
    {'range': 'before_history_start', 'orders': int(eligible_orders['order_approved_at'].lt(HISTORY_START).sum())},
    {'range': 'inside_history', 'orders': len(history_orders)},
    {'range': 'at_or_after_snapshot', 'orders': int(eligible_orders['order_approved_at'].ge(SNAPSHOT_DATE).sum())},
])
display(history_range_counts)
assert history_range_counts['orders'].sum() == len(eligible_orders)
record_stage('09_select_history', history_orders, ['order_id'], '当前快照历史订单',
             excluded_rows=len(eligible_orders)-len(history_orders),
             note='仅表示从此快照历史范围排除；全量订单事实表保留')
assert history_orders['order_approved_at'].ge(HISTORY_START).all()
assert history_orders['order_approved_at'].lt(SNAPSHOT_DATE).all()

## 11 按 customer_unique_id 聚合为客户历史记录

现在才把粒度从订单改成客户快照。F 数不同订单，M 加商品金额，R 是快照距最近批准订单的天数。freight_sum 加历史运费，first/last 记录窗口内的最早与最近事件。

`snapshot_date` 加到每行上之后，主键是 customer_unique_id 与 snapshot_date。即使本轮只有一个日期，也保留这个组合键，为以后多期数据接入做准备。

这份基础历史文件提供 RFM 与追溯字段。其余正式特征在下一阶段根据字段可用性和团队方案完善。

In [ ]:
history_customers = history_orders.groupby('customer_unique_id', as_index=False).agg(
    F=('order_id', 'nunique'),
    M=('amount', 'sum'),
    freight_sum=('freight', 'sum'),
    history_first_event=('order_approved_at', 'min'),
    history_last_event=('order_approved_at', 'max'),
)
history_customers['R'] = (SNAPSHOT_DATE-history_customers['history_last_event']).dt.total_seconds()/86400
history_customers['snapshot_date'] = str(SNAPSHOT_DATE.date())
history_customers['history_start'] = str(HISTORY_START.date())
history_customers['split'] = 'exploration_demo'
history_customers['data_version'] = DATA_VERSION
history_customers['coverage_status'] = 'daily_monthly_coverage_pending'
history_customers = history_customers[[
    'customer_unique_id','snapshot_date','split','data_version','history_start',
    'R','F','M','freight_sum','history_first_event','history_last_event','coverage_status'
]].sort_values('customer_unique_id').reset_index(drop=True)

record_stage('10_aggregate_customer_snapshot', history_customers,
             ['customer_unique_id','snapshot_date'], '客户快照',
             collapsed_rows=len(history_orders)-len(history_customers),
             note='多个历史订单归为一个客户快照；无客户被额外删除')
assert not history_customers.duplicated(['customer_unique_id','snapshot_date']).any()
assert history_customers['F'].ge(1).all()
assert history_customers['F'].sum() == len(history_orders)
assert np.isclose(history_customers['M'].sum(), history_orders['amount'].sum(), rtol=1e-12, atol=1e-7)
assert np.isclose(history_customers['freight_sum'].sum(), history_orders['freight'].sum(), rtol=1e-12, atol=1e-7)
assert np.isfinite(history_customers[['R','F','M','freight_sum']].to_numpy()).all()
display(history_customers.head())
print('历史订单', len(history_orders), '→ 客户快照', len(history_customers))

本快照应得到 **30,134 笔历史合格订单 → 29,368 个客户快照**。差额来自同一买家在窗口内有多笔订单，不是又删掉了 766 个客户。

## 12 客户筛选前后数量及一个多单客户

为了看出金额规则对客户名单的影响，先统计同一历史窗口中所有已知批准时间的买家，再与正式事件规则下的历史客户比较。本步不看未来复购结果。

然后选择一个至少两笔历史合格订单的客户，从订单事实表追溯到原始商品明细，并独立复算 F、M、R。你可以用这段输出手工验证。

In [ ]:
approved_history = approved_orders.loc[
    approved_orders['order_approved_at'].ge(HISTORY_START) & approved_orders['order_approved_at'].lt(SNAPSHOT_DATE)
]
buyers_before_amount = set(approved_history['customer_unique_id'])
buyers_after_amount = set(history_customers['customer_unique_id'])
assert buyers_after_amount.issubset(buyers_before_amount)
customer_filter_counts = pd.DataFrame([
    {'stage': '历史有已知批准时间订单', 'orders': len(approved_history), 'customers': len(buyers_before_amount)},
    {'stage': '历史有批准且正商品金额订单', 'orders': len(history_orders), 'customers': len(buyers_after_amount)},
])
display(customer_filter_counts)
print('因为历史金额资格失去入样资格的买家', len(buyers_before_amount-buyers_after_amount))

example_buyer = history_customers.loc[history_customers['F'].ge(2), 'customer_unique_id'].iloc[0]
buyer_history = history_orders.loc[history_orders['customer_unique_id'].eq(example_buyer)].copy()
buyer_item_details = items.loc[items['order_id'].isin(buyer_history['order_id'])].copy()
buyer_snapshot = history_customers.loc[history_customers['customer_unique_id'].eq(example_buyer)].iloc[0]
display(buyer_history[['order_id','order_approved_at','amount','freight','item_rows']])
display(buyer_item_details[['order_id','order_item_id','price','freight_value']])
display(buyer_snapshot.to_frame('value'))

manual_F = buyer_history['order_id'].nunique()
manual_M = buyer_item_details['price'].sum()
manual_R = (SNAPSHOT_DATE-buyer_history['order_approved_at'].max()).total_seconds()/86400
assert manual_F == buyer_snapshot['F']
assert np.isclose(manual_M, buyer_snapshot['M'], rtol=1e-12, atol=1e-7)
assert np.isclose(manual_R, buyer_snapshot['R'], rtol=1e-12, atol=1e-10)
print('从原始明细和历史订单重算客户 F、M、R 通过。')

## 13 导出本步文件并检查回读

CSV 统一使用 UTF-8 BOM，ID 保持字符串。订单事实表含原始最终状态和交付日期，供追溯检查使用；客户历史表仅含本次已明确的历史聚合字段。

输出清单包括全量订单事实表、演示历史客户表、逐阶段台账、连接缺口、排除订单、历史范围和人数、金额核对及客户案例。文件名明确带有 step1 或 demo，以便与后续正式 H02 区分。

In [ ]:
output_files = []

def save_table(frame, file_name):
    path = OUTPUT_DIR / file_name
    frame.to_csv(path, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d %H:%M:%S')
    output_files.append(path)

order_fact['data_version'] = DATA_VERSION
save_table(order_fact, 'order_fact_audit.csv')
save_table(history_customers, 'history_customers_demo.csv')
save_table(pd.DataFrame(stage_log), 'stage-audit.csv')
save_table(source_checks, 'source-checks.csv')
save_table(relation_checks, 'relation-checks.csv')
save_table(amount_input_check, 'amount-input-checks.csv')
save_table(pd.DataFrame(amount_reconciliation), 'amount-reconciliation.csv')
save_table(excluded_orders, 'excluded-orders.csv')
save_table(history_range_counts, 'history-range-counts.csv')
save_table(customer_filter_counts, 'customer-filter-counts.csv')
save_table(pd.DataFrame(order_trace_checks), 'order-trace-checks.csv')
save_table(buyer_history[['order_id','customer_unique_id','order_approved_at','amount','freight','item_rows']], 'example-buyer-orders.csv')
save_table(buyer_item_details, 'example-buyer-items.csv')
save_table(history_customers.loc[history_customers['customer_unique_id'].eq(example_buyer)], 'example-buyer-snapshot.csv')

saved_orders = pd.read_csv(OUTPUT_DIR / 'order_fact_audit.csv', dtype={'order_id':'string','customer_unique_id':'string'})
saved_customers = pd.read_csv(OUTPUT_DIR / 'history_customers_demo.csv', dtype={'customer_unique_id':'string','snapshot_date':'string'})
assert len(saved_orders) == len(order_fact) and saved_orders['order_id'].is_unique
assert set(saved_orders['order_id']) == set(order_fact['order_id'])
assert saved_orders['amount'].isna().sum() == order_fact['amount'].isna().sum()
assert np.isclose(saved_orders['amount'].sum(), items['price'].sum(), rtol=1e-12, atol=1e-7)
assert not saved_customers.duplicated(['customer_unique_id','snapshot_date']).any()
assert saved_customers['customer_unique_id'].tolist() == history_customers['customer_unique_id'].tolist()
np.testing.assert_allclose(saved_customers[['R','F','M','freight_sum']], history_customers[['R','F','M','freight_sum']], rtol=1e-12, atol=1e-8)
for check in source_checks.to_dict('records'):
    assert file_sha256(DATA_DIR/check['file']) == check['sha256'], '原件在执行过程中发生变化'
print('导出回读核对通过；三个原始 CSV 指纹保持一致。')

## 14 保存本步字段字典与版本清单

清单记录原始包、三个输入文件、本地协议、代码、运行环境和所有产物的哈希。代码版本使用本 Notebook 所有代码格的源代码哈希，避免执行计数与输出变化干扰版本判断。

本步没有拟定正式 train/validation/final 划分。清单会记录哪些部分完成、哪些仍待下一步核定。

In [ ]:
field_dictionary = """# 第一步字段说明

版本 role01-step1-v1-exploration。用途为订单加工与已探索日期的历史聚合教学。

## order_fact_audit.csv

一行一订单，主键 order_id。保留订单原始列、customer_id 和 customer_unique_id。
amount 为按订单汇总的商品 price，单位 BRL。freight 为按订单汇总运费，item_rows 为明细条数。
customer_state 来自客户表。item_match 与 customer_match 分别记录两次左连接匹配结果。
missing_item_amount 表示无法从明细得到金额。eligible_order 表示批准时间存在且商品金额大于零。
exclusion_reason 优先记录 missing_approval，再 missing_item_amount，再 nonpositive_item_amount。
order_status、实际交付日期等供审计，未来交付结果不作为本次客户历史输入。

## history_customers_demo.csv

主键 customer_unique_id 加 snapshot_date，snapshot_date 为 2018-01-01。
history_start 为 2017-07-01，区间含左不含右。split 为 exploration_demo。
R 为快照距最近历史批准事件的秒数除以 86400，保留小数天。
F 为历史不同订单数。M 为历史商品金额之和，BRL，不含运费。
freight_sum 为历史订单运费之和，BRL。
history_first_event 和 history_last_event 为窗口内最早与最近批准时间，供追溯。
data_version 标明版本。coverage_status 为 daily_monthly_coverage_pending。
本表没有 y；其余正式特征及标签在后续步骤核定。

## 台账的数量

rows 是该阶段的行数，grain 表示一行是什么。
unmatched_rows 是连接未匹配行数，非连接阶段留空。
excluded_rows 是该阶段选择排除的行数，依据 grain 解读单位。
collapsed_rows 是聚合前后行数差，不代表删除错误记录。
buyers 是当前表中不同 customer_unique_id 数，尚无该字段时留空。

## 使用边界

本次只核了历史时间条件和连接金额，还需每日／每月覆盖检查。
批准后金额等字段采用当前静态导出的值，缺少源系统字段修改历史。
正式 H02 还需核定候选日期、完善特征与未来标签、执行边界测试和标签开放管理。
"""
dictionary_path = OUTPUT_DIR / 'step1-data-dictionary.md'
dictionary_path.write_text(field_dictionary, encoding='utf-8')
output_files.append(dictionary_path)

notebook_json = json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
code_text = '\n\n'.join(''.join(cell['source']) for cell in notebook_json['cells'] if cell['cell_type']=='code')
run_command = '.\\.venv312\\Scripts\\python.exe -m jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=olist312 notebooks/02-role01-order-fact-and-history.ipynb'
step_manifest = {
    'data_version': DATA_VERSION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'scope': 'step1_order_fact_and_one_exploratory_history_snapshot',
    'formal_H02_status': 'not_complete_pending_steps2_and3_and_team_time_design',
    'source_archive_sha256': archive_sha256,
    'source_files': source_checks.to_dict('records'),
    'local_reference_sha256': reference_hashes,
    'notebook_code_sha256': hashlib.sha256(code_text.encode('utf-8')).hexdigest(),
    'runtime': {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__},
    'definition': {'event': 'approval_present_and_positive_item_amount',
                   'history_start': str(HISTORY_START.date()), 'snapshot_date': str(SNAPSHOT_DATE.date()),
                   'history_interval': '[start,snapshot)', 'split': 'exploration_demo',
                   'history_months': HISTORY_MONTHS, 'labels_generated': False},
    'counts': {'raw_orders': len(orders_raw), 'raw_items': len(items),
               'order_fact': len(order_fact), 'eligible_orders': len(eligible_orders),
               'history_orders': len(history_orders), 'history_customers': len(history_customers),
               'orders_without_items': item_unmatched, 'orders_without_customers': customer_unmatched},
    'checks': {'keys': 'passed', 'relations': 'passed_with_documented_item_gaps',
               'amount_reconciliation': 'passed', 'order_trace_cases': len(order_trace_checks),
               'multi_order_customer_trace': 'passed', 'csv_roundtrip': 'passed', 'raw_inputs_unchanged': True},
    'pending': ['daily_monthly_coverage', 'formal_snapshot_and_evaluation_dates',
                'remaining_feature_contract', 'future_labels_and_maturity',
                'boundary_cases_and_label_release_control', 'team_receipt'],
    'generation_command': run_command,
    'artifacts': [{'file': path.name, 'bytes': path.stat().st_size, 'sha256': file_sha256(path)} for path in output_files],
}
(OUTPUT_DIR / 'step1-manifest.json').write_text(json.dumps(step_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(stage_log))
print('本步完成', step_manifest['counts'])
print('从项目根目录复跑：')
print(run_command)

## 15 你现在检查自己是否理解

1. 商品明细从 112,650 行变成 98,666 行，为什么排除量是零？
2. 左连接后仍有 99,441 行，为什么其中 775 行商品金额是 NA？
3. 为什么订单接客户用 customer_id，而客户聚合用 customer_unique_id？
4. 775 笔无明细订单为什么在第二次资格筛选只排除 629 笔？
5. 2018-01-01 零点发生的批准事件，是否进入本次历史？
6. 为什么历史 30,134 笔订单对应 29,368 个客户，而不是 30,134 个客户？
7. 把第 12 节多单客户的两项金额手算一遍。你能否从原始明细得到同一个 M？

下一步先检查这份 Notebook 的第 5 至 7 节和第 12 节，确认你能解释一次汇总、两次连接和一次客户聚合。之后进入每日／每月覆盖、正式候选日期核定，再生成未来标签和完整 H02。

本次 Notebook、字段说明和产物属于第 1 项的阶段成果。正式 H02 的开放标签、完整覆盖与评价安排仍需后续完成。团队交接时将这些待核项写入交接单。